# 14. Which inputs does Model C rely on? (permutation importance)
**Question:** how much does Model C's validation error grow when one input (or a group of inputs) is scrambled?
**Method:** rebuild the features as in notebook 03, train Model C with the frozen settings, then on the validation hours shuffle the chosen columns (3 shuffles, seed 0) and measure the RMSLE increase. Groups: the three lags together, each lag, calendar, building facts, weather, and every single feature.
**Writes:** `results/importance/permutation_importance_model_C_val.csv`. Takes a few minutes (it trains Model C once).
**Reported result (master 10.11):** baseline validation RMSLE 0.1772; the three lags together +1.961 (1107%), lag_1 +1.815, lag_168 +0.166, building facts +0.159, lag_24 +0.118, calendar +0.039, weather +0.015. Scrambling the previous hour hurts by far the most, so Model C leans on the previous hour.

In [ ]:
from config import *
import numpy as np, pandas as pd
RESULTS = ROOT / "results" / "importance"
RESULTS.mkdir(parents=True, exist_ok=True)
O = str(OUT_DIR) + "/"          # results of notebooks 01-10 (run those first)
def read_any(name):
    """Read name.csv.gz or name.csv or name.parquet from OUT_DIR."""
    for ext in (".csv.gz", ".csv", ".parquet"):
        p = OUT_DIR / (name + ext)
        if p.exists():
            return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
    raise FileNotFoundError(name)

In [ ]:
# This notebook re-uses the feature building and model fitting of notebook 03 (cells up to the point where predictions are saved).
# Its own output folder is redirected so nothing of notebook 03's is overwritten.
import json, config
import shutil
config.OUT_DIR = RESULTS / "_scratch"; config.OUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(OUT_DIR / "labeled_building_ids.csv", config.OUT_DIR)   # the only file of notebook 01 that notebook 03 reads
nb03 = json.load(open("03_gepiii_predictors.ipynb"))
ns = {"__name__": "__main__"}
def run_nb03(stop_when):
    for c in nb03["cells"]:
        if c["cell_type"] != "code": continue
        s = "".join(c["source"])
        if stop_when(s): break
        exec(compile(s, "nb03", "exec"), ns)

In [ ]:
run_nb03(lambda s: 'to_parquet' in s and 'predictor_settings' in s)   # stop before notebook 03 saves anything

In [ ]:
df=ns['df'];FEAT=ns['FEATURES']['C'];m=ns['models']['C']
val=ns['val_rows'].copy()
y=np.log1p(val['y'])
def err(X): return float(np.sqrt(np.mean((m.predict(X[FEAT])-y)**2)))
base=err(val);print('baseline val RMSLE (C):',round(base,4),'rows',len(val))
groups={'lag_1 (previous hour)':['lag_1'],'lag_24 (same hour yesterday)':['lag_24'],'lag_168 (same hour last week)':['lag_168'],
 'calendar: hour, day of week, weekend':['hour','dow','is_weekend'],
 'building facts: id, site, use, size, age, floors':['bld_idx','site_id','primary_use_code','square_feet','year_built','floor_count'],
 'weather (6 columns)':[c for c in FEAT if c in ('air_temperature','dew_temperature','cloud_coverage','precip_depth_1_hr','sea_level_pressure','wind_speed')],
 'all three lags together':['lag_1','lag_24','lag_168']}
for f in FEAT: groups.setdefault('single: '+f,[f])
rng=np.random.default_rng(0);rows=[]
for name,cols in groups.items():
    inc=[]
    for rep in range(3):
        X=val.copy();idx=rng.permutation(len(X))
        for c in cols: X[c]=X[c].to_numpy()[idx]
        inc.append(err(X)-base)
    rows.append(dict(group=name,features=';'.join(cols),rmsle_increase_mean=round(np.mean(inc),4),rmsle_increase_sd=round(np.std(inc),4),relative_increase_pct=round(100*np.mean(inc)/base,1)))
    print(rows[-1])
R=pd.DataFrame(rows).sort_values('rmsle_increase_mean',ascending=False)
R.insert(0,'baseline_rmsle',round(base,4))
R.to_csv(RESULTS/'permutation_importance_model_C_val.csv',index=False)

In [ ]:
# also: does the retrained model reproduce the saved test predictions?
pc = read_any('predictions')[['building_id','timestamp','pred_C']]; pc['timestamp']=pd.to_datetime(pc.timestamp)
t=df[df.split=='test'][['building_id','timestamp','pred_C']].merge(pc,on=['building_id','timestamp'],suffixes=('_new','_saved')).dropna()
print('retrained vs saved pred_C on test: max abs diff',float((t.pred_C_new-t.pred_C_saved).abs().max()))